In [2]:
from pathlib import Path

files = [
    "dialogues_train.txt",
    "dialogues_validation.txt",
    "dialogues_test.txt"
]

for filename in files:
    path = Path("/content") / filename

    with open(path, "r", encoding="utf-8") as f:
        dialogues = [line.strip() for line in f if line.strip()]

    print(f"\n{filename}")
    print(f"Number of dialogues: {len(dialogues)}")
    print(f"First dialogue length: {len(dialogues[0])} characters")
    print(f"Number of utterances: {dialogues[0].count('__eou__')}")


dialogues_train.txt
Number of dialogues: 11118
First dialogue length: 798 characters
Number of utterances: 10

dialogues_validation.txt
Number of dialogues: 1000
First dialogue length: 327 characters
Number of utterances: 7

dialogues_test.txt
Number of dialogues: 1000
First dialogue length: 630 characters
Number of utterances: 12


In [3]:

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("/content")

def load_dialogues(filename):
    path = DATA_DIR / filename

    with open(path, "r", encoding="utf-8") as file:
        lines = [line.strip() for line in file if line.strip()]

    dialogues = []

    for line in lines:
        utterances = [
            utterance.strip()
            for utterance in line.split("__eou__")
            if utterance.strip()
        ]

        if len(utterances) >= 2:
            dialogues.append(utterances)

    return dialogues


train_dialogues = load_dialogues("dialogues_train.txt")
val_dialogues = load_dialogues("dialogues_validation.txt")
test_dialogues = load_dialogues("dialogues_test.txt")

for split_name, dialogues in [
    ("Train", train_dialogues),
    ("Validation", val_dialogues),
    ("Test", test_dialogues),
]:
    lengths = [len(dialogue) for dialogue in dialogues]

    print(f"\n{split_name}")
    print(f"Dialogues: {len(dialogues)}")
    print(f"Average utterances: {np.mean(lengths):.2f}")
    print(f"Median utterances: {np.median(lengths):.0f}")
    print(f"Maximum utterances: {max(lengths)}")

print("\nExample dialogue:")
for i, utterance in enumerate(train_dialogues[0], start=1):
    print(f"{i}. {utterance}")



Train
Dialogues: 11118
Average utterances: 7.84
Median utterances: 7
Maximum utterances: 35

Validation
Dialogues: 1000
Average utterances: 8.07
Median utterances: 8
Maximum utterances: 31

Test
Dialogues: 1000
Average utterances: 7.74
Median utterances: 7
Maximum utterances: 26

Example dialogue:
1. Say , Jim , how about going for a few beers after dinner ?
2. You know that is tempting but is really not good for our fitness .
3. What do you mean ? It will help us to relax .
4. Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ?
5. I guess you are right.But what shall we do ? I don't feel like sitting at home .
6. I suggest a walk over to the gym where we can play singsong and meet some of our friends .
7. That's a good idea . I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome with them .
8. Sounds great to me ! If they are willing , we could ask them to go dancing with us.That is excellent exercise and fun

In [4]:

def create_input_target_pairs(dialogues, max_context_utterances=3):
    pairs = []

    for dialogue in dialogues:
        for i in range(1, len(dialogue)):
            start = max(0, i - max_context_utterances)

            context = dialogue[start:i]
            response = dialogue[i]

            input_text = " ".join(context)

            pairs.append({
                "input": input_text,
                "target": response
            })

    return pairs


train_pairs = create_input_target_pairs(train_dialogues)
val_pairs = create_input_target_pairs(val_dialogues)
test_pairs = create_input_target_pairs(test_dialogues)

print("Training examples:", len(train_pairs))
print("Validation examples:", len(val_pairs))
print("Test examples:", len(test_pairs))

print("\nExample Input:")
print(train_pairs[0]["input"])

print("\nExpected Target:")
print(train_pairs[0]["target"])


Training examples: 76052
Validation examples: 7069
Test examples: 6740

Example Input:
Say , Jim , how about going for a few beers after dinner ?

Expected Target:
You know that is tempting but is really not good for our fitness .


In [5]:
import re
from collections import Counter

def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower())

print(tokenize(train_pairs[0]["input"]))

['say', ',', 'jim', ',', 'how', 'about', 'going', 'for', 'a', 'few', 'beers', 'after', 'dinner', '?']


In [6]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<sos>", "<eos>"]

token_counts = Counter()

for pair in train_pairs:
    token_counts.update(tokenize(pair["input"]))
    token_counts.update(tokenize(pair["target"]))

vocab = {token: idx for idx, token in enumerate(SPECIAL_TOKENS)}

for token, count in token_counts.items():
    if token not in vocab:
        vocab[token] = len(vocab)

stoi = vocab
itos = {idx: token for token, idx in stoi.items()}

PAD_IDX = stoi["<pad>"]
UNK_IDX = stoi["<unk>"]
SOS_IDX = stoi["<sos>"]
EOS_IDX = stoi["<eos>"]

print("Vocabulary size:", len(vocab))
print("Special tokens:", SPECIAL_TOKENS)

Vocabulary size: 17710
Special tokens: ['<pad>', '<unk>', '<sos>', '<eos>']


In [7]:
def encode(text):
    tokens = tokenize(text)
    return [stoi.get(token, UNK_IDX) for token in tokens]

def decode(token_ids):
    return [itos.get(idx, "<unk>") for idx in token_ids]

sample = train_pairs[0]["input"]
encoded = encode(sample)

print("Tokens:", tokenize(sample))
print("Token IDs:", encoded)
print("Decoded:", decode(encoded))

Tokens: ['say', ',', 'jim', ',', 'how', 'about', 'going', 'for', 'a', 'few', 'beers', 'after', 'dinner', '?']
Token IDs: [4, 5, 6, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
Decoded: ['say', ',', 'jim', ',', 'how', 'about', 'going', 'for', 'a', 'few', 'beers', 'after', 'dinner', '?']


In [8]:
MAX_SRC_LEN = 60
MAX_TGT_LEN = 40

def encode_source(text):
    ids = encode(text)[:MAX_SRC_LEN - 1]
    return ids + [EOS_IDX]

def encode_target(text):
    ids = encode(text)[:MAX_TGT_LEN - 1]
    return [SOS_IDX] + ids + [EOS_IDX]

src = encode_source(train_pairs[0]["input"])
tgt = encode_target(train_pairs[0]["target"])

print("Source IDs:", src)
print("Target IDs:", tgt)
print("Source length:", len(src))
print("Target length:", len(tgt))

Source IDs: [4, 5, 6, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 3]
Target IDs: [2, 17, 18, 19, 20, 21, 22, 20, 23, 24, 25, 10, 26, 27, 28, 3]
Source length: 15
Target length: 16


In [9]:
import torch
from torch.nn.utils.rnn import pad_sequence

def collate_batch(batch):
    sources = [
        torch.tensor(encode_source(item["input"]), dtype=torch.long)
        for item in batch
    ]

    targets = [
        torch.tensor(encode_target(item["target"]), dtype=torch.long)
        for item in batch
    ]

    sources = pad_sequence(
        sources, batch_first=True, padding_value=PAD_IDX
    )

    targets = pad_sequence(
        targets, batch_first=True, padding_value=PAD_IDX
    )

    return sources, targets

sample_batch = train_pairs[:4]
src_batch, tgt_batch = collate_batch(sample_batch)

print("Source batch shape:", src_batch.shape)
print("Target batch shape:", tgt_batch.shape)
print(src_batch)
print(tgt_batch)

Source batch shape: torch.Size([4, 52])
Target batch shape: torch.Size([4, 27])
tensor([[ 4,  5,  6,  5,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,  3,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 4,  5,  6,  5,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20,
         21, 22, 20, 23, 24, 25, 10, 26, 27, 28,  3,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 4,  5,  6,  5,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20,
         21, 22, 20, 23, 24, 25, 10, 26, 27, 28, 29, 30, 17, 31, 16, 32, 33, 34,
         35, 36, 37, 28,  3,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [17, 18, 19, 20, 21, 22, 20, 23, 24, 25, 10, 26, 27, 28, 29, 30, 17, 31,
         16, 32, 33, 34, 35, 36, 37, 28, 30, 17, 23, 38, 39, 16, 40, 41, 42, 43,
         28, 32, 33, 44, 45, 35, 46, 47, 48, 49, 2

In [10]:
from torch.utils.data import Dataset, DataLoader

class DialogueDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]


BATCH_SIZE = 32

train_loader = DataLoader(
    DialogueDataset(train_pairs),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

val_loader = DataLoader(
    DialogueDataset(val_pairs),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

test_loader = DataLoader(
    DialogueDataset(test_pairs),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

src_batch, tgt_batch = next(iter(train_loader))

print("Source shape:", src_batch.shape)
print("Target shape:", tgt_batch.shape)
print("Number of training batches:", len(train_loader))

Source shape: torch.Size([32, 60])
Target shape: torch.Size([32, 41])
Number of training batches: 2377


In [11]:
src_batch, tgt_batch = next(iter(train_loader))

decoder_input = tgt_batch[:, :-1]
expected_output = tgt_batch[:, 1:]

print("Decoder input:", decoder_input.shape)
print("Expected output:", expected_output.shape)

Decoder input: torch.Size([32, 40])
Expected output: torch.Size([32, 40])


In [12]:
import math
import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
D_MODEL = 256
NUM_HEADS = 8
D_FF = 1024
MAX_LEN = max(MAX_SRC_LEN, MAX_TGT_LEN) + 1

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        positions = torch.arange(max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(positions * div_term)
        pe[:, 1::2] = torch.cos(positions * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, d_model, padding_idx=pad_idx
        )
        self.scale = math.sqrt(d_model)

    def forward(self, x):
        return self.embedding(x) * self.scale


embedding = TokenEmbedding(len(vocab), D_MODEL, PAD_IDX).to(DEVICE)
pos_encoding = PositionalEncoding(D_MODEL, MAX_LEN).to(DEVICE)

sample = src_batch.to(DEVICE)
x = embedding(sample)
x = pos_encoding(x)

print("Embedding shape:", x.shape)
print("Device:", DEVICE)

Embedding shape: torch.Size([32, 60, 256])
Device: cuda


In [13]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()

        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value,
                key_padding_mask=None, causal=False):

        batch_size = query.size(0)

        def split_heads(x):
            return x.view(
                batch_size, -1, self.num_heads, self.head_dim
            ).transpose(1, 2)

        q = split_heads(self.q_proj(query))
        k = split_heads(self.k_proj(key))
        v = split_heads(self.v_proj(value))

        scores = torch.matmul(q, k.transpose(-2, -1))
        scores = scores / math.sqrt(self.head_dim)

        if key_padding_mask is not None:
            scores = scores.masked_fill(
                key_padding_mask[:, None, None, :],
                float("-inf")
            )

        if causal:
            q_len, k_len = scores.size(-2), scores.size(-1)
            causal_mask = torch.triu(
                torch.ones(
                    q_len, k_len, device=scores.device,
                    dtype=torch.bool
                ),
                diagonal=1
            )
            scores = scores.masked_fill(
                causal_mask[None, None, :, :],
                float("-inf")
            )

        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        output = torch.matmul(attention_weights, v)
        output = output.transpose(1, 2).contiguous()
        output = output.view(batch_size, -1,
                             self.num_heads * self.head_dim)

        return self.out_proj(output)


attention = MultiHeadAttention(D_MODEL, NUM_HEADS).to(DEVICE)

output = attention(
    x, x, x,
    key_padding_mask=(sample == PAD_IDX)
)

print("Attention output shape:", output.shape)

Attention output shape: torch.Size([32, 60, 256])


In [14]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_padding_mask):
        attn = self.attention(
            x, x, x,
            key_padding_mask=src_padding_mask
        )
        x = self.norm1(x + self.dropout(attn))

        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x

In [15]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        self.self_attention = MultiHeadAttention(
            d_model, num_heads, dropout
        )
        self.cross_attention = MultiHeadAttention(
            d_model, num_heads, dropout
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(
        self, x, memory, tgt_padding_mask, src_padding_mask
    ):
        attn = self.self_attention(
            x, x, x,
            key_padding_mask=tgt_padding_mask,
            causal=True
        )
        x = self.norm1(x + self.dropout(attn))

        attn = self.cross_attention(
            x, memory, memory,
            key_padding_mask=src_padding_mask
        )
        x = self.norm2(x + self.dropout(attn))

        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn_out))

        return x

In [16]:
class TransformerChatbot(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=256,
        num_heads=8,
        d_ff=1024,
        num_encoder_layers=3,
        num_decoder_layers=3,
        dropout=0.1,
        max_len=MAX_LEN,
        pad_idx=PAD_IDX
    ):
        super().__init__()

        self.pad_idx = pad_idx

        self.embedding = TokenEmbedding(
            vocab_size, d_model, pad_idx
        )
        self.pos_encoding = PositionalEncoding(d_model, max_len)

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_encoder_layers)
        ])

        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_decoder_layers)
        ])

        self.output_layer = nn.Linear(d_model, vocab_size)

    def encode(self, src):
        src_padding_mask = src.eq(self.pad_idx)

        x = self.pos_encoding(self.embedding(src))

        for layer in self.encoder_layers:
            x = layer(x, src_padding_mask)

        return x, src_padding_mask

    def decode(self, tgt, memory, src_padding_mask):
        tgt_padding_mask = tgt.eq(self.pad_idx)

        x = self.pos_encoding(self.embedding(tgt))

        for layer in self.decoder_layers:
            x = layer(
                x, memory, tgt_padding_mask, src_padding_mask
            )

        return self.output_layer(x)

    def forward(self, src, tgt):
        memory, src_padding_mask = self.encode(src)
        return self.decode(tgt, memory, src_padding_mask)

In [17]:
model = TransformerChatbot(
    vocab_size=len(vocab),
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    d_ff=D_FF,
    num_encoder_layers=3,
    num_decoder_layers=3
).to(DEVICE)

src_batch, tgt_batch = next(iter(train_loader))

src_batch = src_batch.to(DEVICE)
tgt_batch = tgt_batch.to(DEVICE)

decoder_input = tgt_batch[:, :-1]
expected_output = tgt_batch[:, 1:]

with torch.no_grad():
    logits = model(src_batch, decoder_input)

print("Logits shape:", logits.shape)
print("Expected shape:", expected_output.shape)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

Logits shape: torch.Size([32, 40, 17710])
Expected shape: torch.Size([32, 40])
Parameters: 14,614,830


In [18]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

optimizer = optim.Adam(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)

print("Device:", DEVICE)
print("Optimizer:", optimizer.__class__.__name__)
print("Loss:", criterion)

Device: cuda
Optimizer: Adam
Loss: CrossEntropyLoss()


In [20]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    total_correct = 0
    total_tokens = 0

    for src, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)

        decoder_input = tgt[:, :-1]
        expected_output = tgt[:, 1:]

        optimizer.zero_grad(set_to_none=True)
        logits = model(src, decoder_input)

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            expected_output.reshape(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        mask = expected_output.ne(PAD_IDX)
        predictions = logits.argmax(dim=-1)

        total_correct += (
            (predictions == expected_output) & mask
        ).sum().item()
        total_tokens += mask.sum().item()
        total_loss += loss.item()

    return total_loss / len(loader), 100 * total_correct / total_tokens


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_tokens = 0

    for src, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)

        decoder_input = tgt[:, :-1]
        expected_output = tgt[:, 1:]

        logits = model(src, decoder_input)

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            expected_output.reshape(-1)
        )

        mask = expected_output.ne(PAD_IDX)
        predictions = logits.argmax(dim=-1)

        total_correct += (
            (predictions == expected_output) & mask
        ).sum().item()
        total_tokens += mask.sum().item()
        total_loss += loss.item()

    return total_loss / len(loader), 100 * total_correct / total_tokens

In [21]:
from pathlib import Path

EPOCHS = 5
best_val_loss = float("inf")

history = {
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": []
}

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion
    )

    val_loss, val_acc = evaluate(
        model, val_loader, criterion
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_accuracy"].append(train_acc)
    history["val_accuracy"].append(val_acc)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Accuracy: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_acc:.2f}%"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch + 1,
            "val_loss": val_loss
        }, checkpoint_path)

        print("Best model saved.")

Epoch 1/5 | Train Loss: 3.7457 | Train Accuracy: 33.24% | Val Loss: 3.6522 | Val Accuracy: 34.58%
Best model saved.
Epoch 2/5 | Train Loss: 3.5979 | Train Accuracy: 34.58% | Val Loss: 3.5643 | Val Accuracy: 35.47%
Best model saved.
Epoch 3/5 | Train Loss: 3.4839 | Train Accuracy: 35.65% | Val Loss: 3.4997 | Val Accuracy: 36.10%
Best model saved.
Epoch 4/5 | Train Loss: 3.3917 | Train Accuracy: 36.57% | Val Loss: 3.4584 | Val Accuracy: 36.70%
Best model saved.
Epoch 5/5 | Train Loss: 3.3075 | Train Accuracy: 37.52% | Val Loss: 3.4332 | Val Accuracy: 37.22%
Best model saved.


In [22]:
EPOCHS_MORE = 10
PATIENCE = 4

best_val_loss = min(history["val_loss"])
bad_epochs = 0

for epoch in range(EPOCHS_MORE):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion
    )

    val_loss, val_acc = evaluate(
        model, val_loader, criterion
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_accuracy"].append(train_acc)
    history["val_accuracy"].append(val_acc)

    print(
        f"Epoch {len(history['train_loss'])} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        bad_epochs = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": len(history["train_loss"]),
            "val_loss": val_loss
        }, checkpoint_path)

        print("Best model saved.")
    else:
        bad_epochs += 1

    if bad_epochs >= PATIENCE:
        print("Early stopping triggered.")
        break

Epoch 6 | Train Loss: 3.2329 | Train Acc: 38.35% | Val Loss: 3.4033 | Val Acc: 37.49%
Best model saved.
Epoch 7 | Train Loss: 3.1663 | Train Acc: 39.19% | Val Loss: 3.3918 | Val Acc: 37.75%
Best model saved.
Epoch 8 | Train Loss: 3.1040 | Train Acc: 40.02% | Val Loss: 3.3781 | Val Acc: 38.05%
Best model saved.
Epoch 9 | Train Loss: 3.0468 | Train Acc: 40.78% | Val Loss: 3.3732 | Val Acc: 38.42%
Best model saved.
Epoch 10 | Train Loss: 2.9976 | Train Acc: 41.59% | Val Loss: 3.3737 | Val Acc: 38.57%
Epoch 11 | Train Loss: 2.9605 | Train Acc: 42.25% | Val Loss: 3.3847 | Val Acc: 38.93%
Epoch 12 | Train Loss: 2.9296 | Train Acc: 42.93% | Val Loss: 3.3991 | Val Acc: 39.13%
Epoch 13 | Train Loss: 2.9030 | Train Acc: 43.54% | Val Loss: 3.3976 | Val Acc: 39.27%
Early stopping triggered.


In [19]:
import os
import torch

checkpoint_path = "/content/best_transformer.pt"

print("Checkpoint exists:", os.path.exists(checkpoint_path))
print("Vocabulary size:", len(vocab))
print("Device:", DEVICE)

Checkpoint exists: True
Vocabulary size: 17710
Device: cuda


In [21]:
model = TransformerChatbot(
    vocab_size=len(vocab),
    d_model=256,
    num_heads=8,
    d_ff=1024,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dropout=0.1,
    max_len=MAX_LEN,
    pad_idx=PAD_IDX
).to(DEVICE)

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded checkpoint from epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["val_loss"])

Loaded checkpoint from epoch: 9
Best validation loss: 3.3731863013220047


In [23]:
@torch.no_grad()
def generate_response(text, max_new_tokens=40, temperature=0.8, top_k=5):
    model.eval()

    src = torch.tensor(
        [encode_source(text)],
        dtype=torch.long,
        device=DEVICE
    )

    memory, src_padding_mask = model.encode(src)

    generated = torch.tensor(
        [[SOS_IDX]],
        dtype=torch.long,
        device=DEVICE
    )

    for _ in range(max_new_tokens):
        logits = model.decode(
            generated, memory, src_padding_mask
        )[:, -1, :]

        logits = logits / temperature
        logits[:, PAD_IDX] = float("-inf")
        logits[:, SOS_IDX] = float("-inf")

        if top_k is not None:
            k = min(top_k, logits.size(-1))
            values, _ = torch.topk(logits, k, dim=-1)
            logits[logits < values[:, -1:]] = float("-inf")

        probabilities = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probabilities, num_samples=1)

        if next_token.item() == EOS_IDX:
            break

        generated = torch.cat([generated, next_token], dim=1)

    tokens = decode(generated[0, 1:].tolist())
    return " ".join(tokens)


test_messages = [
    "hello",
    "how are you?",
    "what do you like to do?",
    "i am feeling sad today.",
    "what is your favorite movie?"

]

for message in test_messages:
    print(f"\nUser: {message}")
    print(f"Bot:  {generate_response(message)}")


User: hello
Bot:  hi , mary . i ' m just looking for an interview for my daughter to call . i ’ ll be back on a business trip to see you .

User: how are you?
Bot:  i ’ m great .

User: what do you like to do?
Bot:  i like to like to live in the country .

User: i am feeling sad today.
Bot:  what ' s the matter ?

User: what is your favorite movie?
Bot:  i like the movies .


In [24]:
@torch.no_grad()
def generate_greedy(text, max_new_tokens=30):
    model.eval()

    src = torch.tensor(
        [encode_source(text)],
        dtype=torch.long,
        device=DEVICE
    )

    memory, src_padding_mask = model.encode(src)

    generated = torch.tensor(
        [[SOS_IDX]],
        dtype=torch.long,
        device=DEVICE
    )

    for _ in range(max_new_tokens):
        logits = model.decode(
            generated, memory, src_padding_mask
        )[:, -1, :]

        logits[:, PAD_IDX] = float("-inf")
        logits[:, SOS_IDX] = float("-inf")

        next_token = logits.argmax(dim=-1).item()

        if next_token == EOS_IDX:
            break

        generated = torch.cat([
            generated,
            torch.tensor([[next_token]], device=DEVICE)
        ], dim=1)

    return " ".join(decode(generated[0, 1:].tolist()))


for message in test_messages:
    print(f"\nUser: {message}")
    print(f"Bot: {generate_greedy(message)}")


User: hello
Bot: hi , mary . what ' s your name ?

User: how are you?
Bot: i ' m fine , thanks .

User: what do you like to do?
Bot: i like to go to the movies .

User: i am feeling sad today.
Bot: you are so kidding .

User: what is your favorite movie?
Bot: i like the movies .


In [25]:
import random

random.seed(42)

for item in random.sample(train_pairs, 10):
    print("INPUT :", item["input"])
    print("TARGET:", item["target"])
    print("-" * 80)

INPUT : Excuse me , I need your assistance . Sure , what's up ?
TARGET: I have no idea where my class is .
--------------------------------------------------------------------------------
INPUT : Oh , thank you for pointing that out . By the way , have you been following NBA for this season ? Of course , tomorrow evening there will be a most exciting game . Who plays who ?
TARGET: Rockets vs Lakers . Yao Ming , my idol , will appear in front of several hundred million of the worldwide audience .
--------------------------------------------------------------------------------
INPUT : Well , what do you have in mind ? How about the Mongolian hot pot ? Oh , it's too spicy for me . Don't you remember last time when I tasted the lamb ? It Was so spicy my eyes teared up !
TARGET: Maybe we can try ' Yuanyang pot ' this time . You can choose the non-spicy soup base .
--------------------------------------------------------------------------------
INPUT : don ’ t worry . I don't think you ’ re 

In [26]:
import re

def tokenize(text):
    text = text.lower()
    text = text.replace("’", "'").replace("‘", "'")
    return re.findall(
        r"\w+(?:'\w+)*|[^\w\s]",
        text
    )

print(tokenize("I'm fine, thanks! I don't know."))

["i'm", 'fine', ',', 'thanks', '!', 'i', "don't", 'know', '.']


In [29]:
import torch
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = Path("/content/best_transformer.pt")

print("Device:", DEVICE)
print("Checkpoint exists:", checkpoint_path.exists())

if checkpoint_path.exists():
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )

    print("Saved epoch:", checkpoint["epoch"])
    print("Best validation loss:", checkpoint["val_loss"])

Device: cuda
Checkpoint exists: True
Saved epoch: 9
Best validation loss: 3.3731863013220047


In [30]:
import torch

model = TransformerChatbot(
    vocab_size=len(stoi),
    d_model=256,
    num_heads=8,
    d_ff=1024,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dropout=0.1,
    max_len=MAX_LEN,
    pad_idx=PAD_IDX
).to(DEVICE)

checkpoint = torch.load(
    "/content/best_transformer.pt",
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model loaded successfully!")
print("Device:", DEVICE)
print("Parameters:", sum(p.numel() for p in model.parameters()))

Model loaded successfully!
Device: cuda
Parameters: 14614830


In [31]:
@torch.no_grad()
def generate_response(text, max_new_tokens=40):
    model.eval()

    src_ids = encode_source(text)

    src = torch.tensor(
        [src_ids],
        dtype=torch.long,
        device=DEVICE
    )

    memory, src_padding_mask = model.encode(src)

    generated = torch.tensor(
        [[SOS_IDX]],
        dtype=torch.long,
        device=DEVICE
    )

    for _ in range(max_new_tokens):
        logits = model.decode(
            generated,
            memory,
            src_padding_mask
        )

        next_token_logits = logits[:, -1, :]
        next_token_logits[:, PAD_IDX] = float("-inf")
        next_token_logits[:, SOS_IDX] = float("-inf")

        next_token = next_token_logits.argmax(dim=-1).item()

        if next_token == EOS_IDX:
            break

        generated = torch.cat(
            [
                generated,
                torch.tensor([[next_token]], device=DEVICE)
            ],
            dim=1
        )

    output_ids = generated[0, 1:].tolist()

    response_tokens = [
        itos[token_id]
        for token_id in output_ids
        if token_id not in {PAD_IDX, SOS_IDX, EOS_IDX}
    ]

    return " ".join(response_tokens)

In [32]:
test_inputs = [
    "Hello!",
    "How are you?",
    "What are you doing?",
    "I am feeling sad today.",
    "What is your favorite movie?"
]

for message in test_inputs:
    print("User:", message)
    print("Bot:", generate_response(message))
    print("-" * 50)

User: Hello!
Bot: hi , mary . i ' m just wondering if you ' re interested in the wrong bus .
--------------------------------------------------
User: How are you?
Bot: i ' m fine , thanks .
--------------------------------------------------
User: What are you doing?
Bot: i ' m just watching tv .
--------------------------------------------------
User: I am feeling sad today.
Bot: you are so kidding .
--------------------------------------------------
User: What is your favorite movie?
Bot: i like the movies .
--------------------------------------------------


In [33]:
import torch

assert torch.cuda.is_available(), "Please enable GPU in Colab settings."

print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print("PyTorch:", torch.__version__)

GPU: Tesla T4
GPU Memory: 14.56 GB
PyTorch: 2.11.0+cu130


In [34]:
%pip install -q -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.3 MB/s eta 0:00:00


In [35]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()

print("Model loaded successfully!")
print("Model:", MODEL_ID)
print("Device:", model.device)
print("Compute dtype:", compute_dtype)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Model: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0
Compute dtype: torch.bfloat16


In [36]:
def generate_response(message, max_new_tokens=150):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful conversational assistant. "
                "Answer naturally, clearly, and stay relevant to the user's message."
            )
        },
        {
            "role": "user",
            "content": message
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

In [37]:
test_inputs = [
    "Hello!",
    "How are you?",
    "What are you doing?",
    "I am feeling sad today.",
    "What is your favorite movie?",
    "Explain machine learning in simple words."
]

for message in test_inputs:
    print(f"User: {message}")
    print(f"Bot:  {generate_response(message)}")
    print("-" * 60)

User: Hello!
Bot:  Hello! How can I help you today? Is there something specific you would like to talk about or ask about? I'm here to answer any questions you might have.
------------------------------------------------------------
User: How are you?
Bot:  As an AI language model, I don't have feelings in the way that humans do, but I'm here to help answer any questions or provide information to the best of my ability. Is there something specific you would like to know or discuss?
------------------------------------------------------------
User: What are you doing?
Bot:  I am here to help answer your questions and have a conversation with you. Is there anything specific you would like to know or discuss?
------------------------------------------------------------
User: I am feeling sad today.
Bot:  It's important to acknowledge when you're feeling down or experiencing sadness. Here are a few ways you might find comfort:

1. Reach out: Talk about your feelings with someone you trust.

In [38]:

import torch
import gradio as gr

SYSTEM_PROMPT = """
You are a helpful conversational assistant.
Answer naturally, clearly, and stay relevant to the conversation.
"""

def chat(message, history):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]

    # Keep previous messages for conversational context
    for item in history:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content", "")

            if role in ("user", "assistant") and isinstance(content, str):
                messages.append({
                    "role": role,
                    "content": content
                })

    messages.append({
        "role": "user",
        "content": message
    })

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


demo = gr.ChatInterface(
    fn=chat,
    title="My Transformer Chatbot",
    description="A conversational AI chatbot powered by Qwen2.5-1.5B-Instruct.",
    examples=[
        "Hello! How are you?",
        "Explain machine learning in simple words.",
        "I'm feeling sad today. Can we talk?",
        "Help me prepare a study plan for AI."
    ]
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8152e6ef9022a0b94b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [40]:

import torch
import gradio as gr

SYSTEM_PROMPT = """
You are a helpful bilingual conversational assistant.

Rules:
- If the user writes in Arabic, reply in natural, fluent Arabic.
- If the user writes in Egyptian Arabic, prefer natural Egyptian Arabic.
- If the user writes in English, reply in English.
- If the user mixes Arabic and English, respond naturally in the same style.
- Understand the conversation context and previous messages.
- Give relevant, clear, and useful answers.
- Do not unnecessarily repeat the user's question.
"""

def chat(message, history):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]

    # Preserve conversation history
    for item in history:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content", "")

            if role in ("user", "assistant") and isinstance(content, str):
                messages.append({
                    "role": role,
                    "content": content
                })

    messages.append({
        "role": "user",
        "content": message
    })

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return response


demo = gr.ChatInterface(
    fn=chat,
    title="المساعد الذكي | AI Chatbot",
    description="مساعد ذكي يتحدث العربية والإنجليزية ويدعم سياق المحادثة.",
    examples=[
        "أهلاً، عامل إيه؟",
        "اشرح لي الفرق بين Machine Learning و Deep Learning.",
        "أنا حاسس بالإحباط من المذاكرة، تنصحني بإيه؟",
        "Explain Transformers in simple English.",
        "إزاي أبدأ أتعلم الذكاء الاصطناعي؟"
    ],
    textbox=gr.Textbox(
        placeholder="اكتب رسالتك هنا... / Type your message",
        container=False
    )
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eecfb40a9e54cbf197.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [41]:

import os
import json
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/content/transformer_chatbot_project")
BASELINE_DIR = PROJECT_DIR / "baselines"
RESULTS_DIR = PROJECT_DIR / "results"
ADAPTER_DIR = PROJECT_DIR / "qwen_daily_dialog_lora"

for folder in [BASELINE_DIR, RESULTS_DIR, ADAPTER_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

evaluation_questions = [
    "Hello! How are you?",
    "What do you like to do in your free time?",
    "I am feeling sad today. Can we talk?",
    "How can I become more confident?",
    "Explain machine learning in simple words.",
    "What should I do when I cannot concentrate?",
    "I had a difficult day at work.",
    "What is the difference between AI and machine learning?",
    "Can you suggest a healthy daily routine?",
    "I want to learn programming. Where should I start?",
    "Why is communication important in relationships?",
    "Tell me an interesting fact about space.",
    "عامل إيه؟",
    "اشرحلي يعني إيه ذكاء اصطناعي بطريقة بسيطة."
]

with open(RESULTS_DIR / "evaluation_questions.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_questions, f, ensure_ascii=False, indent=2)

print("Questions:", len(evaluation_questions))
print("Project directory:", PROJECT_DIR)


Questions: 14
Project directory: /content/transformer_chatbot_project


In [42]:

baseline_results = []

for i, question in enumerate(evaluation_questions, start=1):
    answer = generate_response(question)

    baseline_results.append({
        "question_id": i,
        "question": question,
        "baseline_answer": answer
    })

    print(f"{i}. User: {question}")
    print(f"   Bot: {answer}\n")

baseline_df = pd.DataFrame(baseline_results)

baseline_path = RESULTS_DIR / "qwen_before_finetuning.csv"
baseline_df.to_csv(baseline_path, index=False, encoding="utf-8-sig")

print("Saved baseline:", baseline_path)


1. User: Hello! How are you?
   Bot: Hello! I'm just like you - an AI assistant here to help with your questions or tasks. How can I assist you today?

2. User: What do you like to do in your free time?
   Bot: As an artificial intelligence, I don't have personal preferences or hobbies like humans. However, I'm here to assist with any questions or tasks you might have! Is there anything specific you'd like help with today?

3. User: I am feeling sad today. Can we talk?
   Bot: Of course, I'm here to listen if you need someone to talk to. Let me know how you'd like to proceed.

4. User: How can I become more confident?
   Bot: Becoming more confident is a great goal! Here are some steps you might consider:

1. Set small achievable goals: Start by setting realistic goals that don't overwhelm you. Each time you achieve one of these, it will build your confidence.

2. Practice positive self-talk: Replace negative thoughts with affirmations or encouraging statements about yourself.

3. Take

In [43]:

import shutil

original_checkpoint = Path("/content/best_transformer.pt")

if original_checkpoint.exists():
    shutil.copy2(
        original_checkpoint,
        BASELINE_DIR / "transformer_from_scratch_epoch9.pt"
    )
    print("Original Transformer checkpoint preserved.")
else:
    print("Checkpoint not found. Upload it before making the backup.")


Original Transformer checkpoint preserved.


In [44]:

from pathlib import Path
import random

DATA_DIR = Path("/content")
SYSTEM_PROMPT = (
    "You are a helpful conversational assistant. "
    "Respond naturally and stay relevant to the conversation."
)

def load_dialogues(filename):
    dialogues = []

    with open(DATA_DIR / filename, "r", encoding="utf-8") as f:
        for line in f:
            utterances = [
                x.strip()
                for x in line.strip().split("__eou__")
                if x.strip()
            ]
            if len(utterances) >= 2:
                dialogues.append(utterances)

    return dialogues

train_dialogues = load_dialogues("dialogues_train.txt")
val_dialogues = load_dialogues("dialogues_validation.txt")
test_dialogues = load_dialogues("dialogues_test.txt")

def create_pairs(dialogues, max_context=3):
    pairs = []

    for dialogue in dialogues:
        for i in range(1, len(dialogue)):
            context = dialogue[max(0, i - max_context):i]
            response = dialogue[i]

            pairs.append({
                "context": "\n".join(context),
                "response": response
            })

    return pairs

train_pairs = create_pairs(train_dialogues)
val_pairs = create_pairs(val_dialogues)
test_pairs = create_pairs(test_dialogues)

print("Train pairs:", len(train_pairs))
print("Validation pairs:", len(val_pairs))
print("Test pairs:", len(test_pairs))
print("\nExample:")
print("Context:", train_pairs[0]["context"])
print("Target:", train_pairs[0]["response"])


Train pairs: 76052
Validation pairs: 7069
Test pairs: 6740

Example:
Context: Say , Jim , how about going for a few beers after dinner ?
Target: You know that is tempting but is really not good for our fitness .


In [50]:

sample = train_pairs[0]

prompt_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": sample["context"]}
]

print("PROMPT TEMPLATE:")
print(tokenizer.apply_chat_template(
    prompt_messages,
    tokenize=False,
    add_generation_prompt=True
))

print("\nTARGET RESPONSE:")
print(sample["response"])

print("\nEOS TOKEN:", tokenizer.eos_token)


PROMPT TEMPLATE:
<|im_start|>system
You are a helpful conversational assistant. Respond naturally and stay relevant to the conversation.<|im_end|>
<|im_start|>user
Can I help you , sir ?<|im_end|>
<|im_start|>assistant


TARGET RESPONSE:
Yes , please . I'm picking out some presents for my children and wife .

EOS TOKEN: <|im_end|>


In [51]:

class DialogueSFTDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=512):
        self.rows = []
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.skipped = 0

        for pair in pairs:
            prompt_messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": pair["context"]}
            ]

            prompt_text = tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True
            )

            # Render the prompt and answer as one continuous sequence.
            full_text = prompt_text + pair["response"] + tokenizer.eos_token

            prompt_ids = tokenizer.encode(
                prompt_text,
                add_special_tokens=False
            )

            full_ids = tokenizer.encode(
                full_text,
                add_special_tokens=False
            )

            # Find the common prefix instead of rejecting every mismatch.
            prefix_len = 0
            limit = min(len(prompt_ids), len(full_ids))

            while (
                prefix_len < limit
                and prompt_ids[prefix_len] == full_ids[prefix_len]
            ):
                prefix_len += 1

            # Skip examples only if the prompt cannot be aligned reliably.
            if prefix_len < 5 or prefix_len >= len(full_ids):
                self.skipped += 1
                continue

            full_ids = full_ids[:max_length]
            if prefix_len >= len(full_ids):
                self.skipped += 1
                continue

            labels = [-100] * prefix_len + full_ids[prefix_len:]

            self.rows.append({
                "input_ids": full_ids,
                "attention_mask": [1] * len(full_ids),
                "labels": labels
            })

        print(
            f"Prepared: {len(self.rows)} | "
            f"Skipped: {self.skipped}"
        )

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


In [52]:

train_dataset = DialogueSFTDataset(
    train_subset, tokenizer, max_length=512
)

val_dataset = DialogueSFTDataset(
    val_subset, tokenizer, max_length=512
)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

if len(train_dataset) > 0:
    sample = train_dataset[0]

    print("Example token length:", len(sample["input_ids"]))
    print(
        "Tokens supervised for loss:",
        sum(label != -100 for label in sample["labels"])
    )
else:
    print("Dataset is empty. Do not start training.")


Prepared: 12000 | Skipped: 0
Prepared: 1500 | Skipped: 0
Training examples: 12000
Validation examples: 1500
Example token length: 54
Tokens supervised for loss: 17


In [53]:

sample = train_dataset[0]

input_ids = sample["input_ids"]
labels = sample["labels"]

print("FULL SEQUENCE:")
print(tokenizer.decode(input_ids, skip_special_tokens=False))

print("\nTOKENS INCLUDED IN LOSS:")
supervised_ids = [
    token_id
    for token_id, label in zip(input_ids, labels)
    if label != -100
]
print(tokenizer.decode(supervised_ids, skip_special_tokens=False))

print("\nFIRST SUPERVISED TOKENS:")
print(tokenizer.convert_ids_to_tokens(supervised_ids[:20]))


FULL SEQUENCE:
<|im_start|>system
You are a helpful conversational assistant. Respond naturally and stay relevant to the conversation.<|im_end|>
<|im_start|>user
Can I help you , sir ?<|im_end|>
<|im_start|>assistant
Yes , please . I'm picking out some presents for my children and wife .<|im_end|>

TOKENS INCLUDED IN LOSS:
Yes , please . I'm picking out some presents for my children and wife .<|im_end|>

FIRST SUPERVISED TOKENS:
['Yes', 'Ġ,', 'Ġplease', 'Ġ.', 'ĠI', "'m", 'Ġpicking', 'Ġout', 'Ġsome', 'Ġpresents', 'Ġfor', 'Ġmy', 'Ġchildren', 'Ġand', 'Ġwife', 'Ġ.', '<|im_end|>']


In [54]:
from google.colab import drive
drive.mount("/content/drive")

import os
import json
import shutil
from pathlib import Path

PROJECT_DIR = Path("/content/transformer_chatbot_project")
DRIVE_DIR = Path("/content/drive/MyDrive/transformer_chatbot_project")

# Create folders
for folder in [
    DRIVE_DIR / "baselines",
    DRIVE_DIR / "results",
    DRIVE_DIR / "qwen_daily_dialog_lora"
]:
    folder.mkdir(parents=True, exist_ok=True)

# Save the original from-scratch Transformer checkpoint
scratch_checkpoint = Path("/content/best_transformer.pt")

if scratch_checkpoint.exists():
    shutil.copy2(
        scratch_checkpoint,
        DRIVE_DIR / "baselines" / "transformer_from_scratch_epoch9.pt"
    )
    print("Saved original Transformer checkpoint.")
else:
    print("Original Transformer checkpoint not found.")

# Save Qwen's pre-fine-tuning evaluation results
baseline_csv = PROJECT_DIR / "results" / "qwen_before_finetuning.csv"

if baseline_csv.exists():
    shutil.copy2(
        baseline_csv,
        DRIVE_DIR / "results" / "qwen_before_finetuning.csv"
    )
    print("Saved Qwen baseline evaluation results.")
else:
    print("Baseline CSV not found. Generate it before training.")

print("Drive location:", DRIVE_DIR)


Mounted at /content/drive
Saved original Transformer checkpoint.
Saved Qwen baseline evaluation results.
Drive location: /content/drive/MyDrive/transformer_chatbot_project


In [55]:
from pathlib import Path
import json
import torch

ADAPTER_DIR = Path(
    "/content/drive/MyDrive/transformer_chatbot_project/qwen_daily_dialog_lora"
)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

# Save LoRA adapter and its configuration
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Save training metadata
metadata = {
    "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
    "dataset": "DailyDialog",
    "training_method": "LoRA",
    "quantization": "4-bit NF4",
    "max_context_utterances": 3
}

with open(ADAPTER_DIR / "training_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("LoRA adapter saved to:", ADAPTER_DIR)
print("\nSaved files:")
for file in sorted(ADAPTER_DIR.iterdir()):
    print(f"{file.name} — {file.stat().st_size / (1024**2):.2f} MB")

LoRA adapter saved to: /content/drive/MyDrive/transformer_chatbot_project/qwen_daily_dialog_lora

Saved files:
README.md — 0.00 MB
adapter_config.json — 0.00 MB
adapter_model.safetensors — 35.27 MB
chat_template.jinja — 0.00 MB
tokenizer.json — 10.89 MB
tokenizer_config.json — 0.00 MB
training_metadata.json — 0.00 MB


In [56]:
import shutil
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/transformer_chatbot_project")
ZIP_PATH = "/content/transformer_chatbot_project"

shutil.make_archive(
    ZIP_PATH,
    "zip",
    root_dir=DRIVE_DIR
)

print("ZIP created:", ZIP_PATH + ".zip")

ZIP created: /content/transformer_chatbot_project.zip


In [57]:
from google.colab import files

files.download("/content/transformer_chatbot_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:

%pip install -q -U peft


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 18.5 MB/s eta 0:00:00


In [58]:

import torch
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
from transformers import Trainer, TrainingArguments

# The currently loaded Qwen model is quantized to 4-bit.
model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [59]:

from torch.nn.utils.rnn import pad_sequence

pad_id = tokenizer.pad_token_id
if pad_id is None:
    pad_id = tokenizer.eos_token_id

def collate_fn(features):
    input_ids = [
        torch.tensor(x["input_ids"], dtype=torch.long)
        for x in features
    ]
    attention_masks = [
        torch.tensor(x["attention_mask"], dtype=torch.long)
        for x in features
    ]
    labels = [
        torch.tensor(x["labels"], dtype=torch.long)
        for x in features
    ]

    input_ids = pad_sequence(
        input_ids, batch_first=True, padding_value=pad_id
    )
    attention_masks = pad_sequence(
        attention_masks, batch_first=True, padding_value=0
    )
    labels = pad_sequence(
        labels, batch_first=True, padding_value=-100
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "labels": labels
    }

use_bf16 = torch.cuda.is_bf16_supported()

training_args = TrainingArguments(
    output_dir=str(PROJECT_DIR / "training_checkpoints"),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    fp16=not use_bf16,
    bf16=use_bf16,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    report_to="none",
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn
)

trainer.train()

model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

print("LoRA adapter saved to:", ADAPTER_DIR)


Epoch,Training Loss,Validation Loss
1,2.237820,2.072863


LoRA adapter saved to: /content/drive/MyDrive/transformer_chatbot_project/qwen_daily_dialog_lora


In [60]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = (
    "/content/drive/MyDrive/"
    "transformer_chatbot_project/qwen_daily_dialog_lora"
)

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

# Load a fresh base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
base_model.eval()

# Load the fine-tuned adapter separately
fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)
fine_tuned_model.eval()

print("Base model and LoRA adapter loaded successfully.")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model and LoRA adapter loaded successfully.


/usr/local/lib/python3.13/dist-packages/peft/peft_model.py:642: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_B.default.we

In [61]:

import pandas as pd
import torch
from pathlib import Path

evaluation_questions = [
    "Hello! How are you?",
    "What do you like to do in your free time?",
    "I am feeling sad today. Can we talk?",
    "How can I become more confident?",
    "Explain machine learning in simple words.",
    "What should I do when I cannot concentrate?",
    "I had a difficult day at work.",
    "What is the difference between AI and machine learning?",
    "Can you suggest a healthy daily routine?",
    "I want to learn programming. Where should I start?",
    "Why is communication important in relationships?",
    "Tell me an interesting fact about space.",
    "عامل إيه؟",
    "اشرحلي يعني إيه ذكاء اصطناعي بطريقة بسيطة."
]

SYSTEM_PROMPT = (
    "You are a helpful conversational assistant. "
    "Respond naturally and stay relevant to the conversation. "
    "Reply in the same language used by the user."
)

def generate_with_model(chat_model, question, max_new_tokens=120):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda")

    with torch.inference_mode():
        output = chat_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


results = []

for i, question in enumerate(evaluation_questions, start=1):
    before = generate_with_model(base_model, question)
    after = generate_with_model(fine_tuned_model, question)

    results.append({
        "question_id": i,
        "question": question,
        "before_finetuning": before,
        "after_finetuning": after
    })

    print(f"\n{'=' * 70}")
    print(f"Question {i}: {question}")
    print(f"\nBEFORE:\n{before}")
    print(f"\nAFTER:\n{after}")

results_df = pd.DataFrame(results)

output_path = (
    "/content/drive/MyDrive/transformer_chatbot_project/"
    "results/qwen_before_vs_after.csv"
)

Path(output_path).parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\nComparison saved to:", output_path)



Question 1: Hello! How are you?

BEFORE:
Hello! I'm just an AI, so I don't have feelings or consciousness like humans do. But thank you for asking! How can I help you today?

AFTER:
Hello! I'm just an AI, so I don't have feelings or consciousness like humans do. But thank you for asking! How can I help you today?

Question 2: What do you like to do in your free time?

BEFORE:
As an AI, I don't have personal preferences or experiences like humans do. However, I'm here to help answer questions and provide information on various topics. Is there something specific you'd like to know about?

AFTER:
As an AI, I don't have personal preferences or experiences like humans do. However, I'm here to help answer questions and provide information on various topics. Is there something specific you'd like to know about?

Question 3: I am feeling sad today. Can we talk?

BEFORE:
Of course, I'm here to listen if you need someone to talk to or just want to know that you're not alone. Let me know how yo

In [62]:

from pathlib import Path
import shutil
import json

PROJECT_DIR = Path("/content/transformer_chatbot_project")
DRIVE_DIR = Path(
    "/content/drive/MyDrive/transformer_chatbot_project"
)

# Create project folders
for folder in [
    DRIVE_DIR / "baselines",
    DRIVE_DIR / "results",
    DRIVE_DIR / "qwen_daily_dialog_lora",
    DRIVE_DIR / "original_transformer"
]:
    folder.mkdir(parents=True, exist_ok=True)

# Save original Transformer checkpoint
scratch_checkpoint = Path("/content/best_transformer.pt")

if scratch_checkpoint.exists():
    shutil.copy2(
        scratch_checkpoint,
        DRIVE_DIR / "baselines" /
        "transformer_from_scratch_epoch9.pt"
    )
    print("Original Transformer checkpoint saved.")

# Save evaluation results
for filename in [
    "qwen_before_finetuning.csv",
    "qwen_before_vs_after.csv",
    "evaluation_questions.json"
]:
    source = PROJECT_DIR / "results" / filename

    if source.exists():
        shutil.copy2(source, DRIVE_DIR / "results" / filename)
        print("Saved:", filename)

# Save model information
model_info = {
    "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
    "fine_tuning": "LoRA",
    "quantization": "4-bit NF4",
    "dataset": "DailyDialog",
    "training_examples": 12000,
    "validation_examples": 1500,
    "epochs": 1,
    "training_loss": 2.237820,
    "validation_loss": 2.072863
}

with open(
    DRIVE_DIR / "model_info.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(model_info, f, indent=2, ensure_ascii=False)

print("\nProject directory:", DRIVE_DIR)


Original Transformer checkpoint saved.
Saved: qwen_before_finetuning.csv
Saved: evaluation_questions.json

Project directory: /content/drive/MyDrive/transformer_chatbot_project


In [63]:

import shutil
from pathlib import Path

DRIVE_DIR = Path(
    "/content/drive/MyDrive/transformer_chatbot_project"
)

ZIP_PATH = "/content/transformer_chatbot_project"

archive = shutil.make_archive(
    ZIP_PATH,
    "zip",
    root_dir=DRIVE_DIR
)

print("Archive created:", archive)
print("Archive size (MB):", round(Path(archive).stat().st_size / 1e6, 2))


Archive created: /content/transformer_chatbot_project.zip
Archive size (MB): 197.6


In [64]:

from google.colab import files

files.download("/content/transformer_chatbot_project.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [65]:

!pip -q install gradio peft bitsandbytes accelerate transformers


In [66]:

import gradio as gr
import torch

SYSTEM_PROMPT = """
You are a helpful conversational assistant.
Reply in the same language used by the user.
If the user speaks Egyptian Arabic, reply naturally in Egyptian Arabic.
If the user speaks English, reply in English.
Be friendly, clear, and relevant.
Use the conversation history to maintain context.
"""

def chat(message, history):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]

    # Add previous conversation turns
    for item in history:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")

            if role in ("user", "assistant") and isinstance(content, str):
                messages.append({
                    "role": role,
                    "content": content
                })

        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_message, assistant_message = item

            if user_message:
                messages.append({
                    "role": "user",
                    "content": user_message
                })

            if assistant_message:
                messages.append({
                    "role": "assistant",
                    "content": assistant_message
                })

    messages.append({"role": "user", "content": message})

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda")

    with torch.inference_mode():
        output = fine_tuned_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return answer or "I couldn't generate a response. Please try again."

demo = gr.ChatInterface(
    fn=chat,
    title="DailyDialog AI Chatbot",
    description=(
        "A conversational chatbot powered by "
        "Qwen2.5-1.5B-Instruct fine-tuned with LoRA on DailyDialog. "
        "Supports English and Arabic."
    ),
    textbox=gr.Textbox(
        placeholder="Type your message in Arabic or English...",
        container=False
    ),
    save_history=True
)

demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0a874a974386135aa2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
